# Model Training & Evaluation


In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../data/processed/featured_loans.csv')
df = df.select_dtypes(include=np.number).dropna()
X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']
print('Default rate:', y.mean())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                      use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:,1]
print(f'AUC: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_prob):.3f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve - Loan Default Model')
plt.legend()
plt.show()

In [ ]:
# Feature Importance
feat_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)[:10]
feat_imp.plot(kind='bar', color='steelblue')
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Risk Score Distribution
y_scores = y_prob * 100
plt.figure(figsize=(10,5))
plt.hist(y_scores[y_test==0], bins=50, alpha=0.6, label='No Default', color='steelblue')
plt.hist(y_scores[y_test==1], bins=50, alpha=0.6, label='Default', color='red')
plt.axvline(30, color='green', linestyle='--', label='Low/Medium boundary')
plt.axvline(60, color='orange', linestyle='--', label='Medium/High boundary')
plt.axvline(80, color='red', linestyle='--', label='High/Critical boundary')
plt.xlabel('Risk Score')
plt.title('Risk Score Distribution by Default Status')
plt.legend()
plt.show()